<a href="https://colab.research.google.com/github/Tirantha/Statistical-Learning-e22040/blob/main/e22040%20Data%20wrangling%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine


In [2]:
import pandas as pd
import numpy as np
import io
from google.colab import files
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
import scipy.stats as ss
from IPython.display import display, HTML

class PlottingMethods:

    def __init__(self):
        pass

    def get_bar_chart_html(self, df, x_col, y_col, title="Bar Chart"):
        fig = px.bar(df, x=x_col, y=y_col, title=title)
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    def get_pie_chart_html(self, df, names_col, values_col, title="Pie Chart"):
        fig = px.pie(df, names=names_col, values=values_col, title=title)
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    def get_histogram_html(self, df, x_col, title="Histogram"):
        fig = px.histogram(df, x=x_col, title=title)
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

class DataInspector:

    def __init__(self):
        self.df = None
        self.original_df = None
        self.plotter = PlottingMethods()

    def upload_data(self):
        print("Please upload your CSV file:")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return

        filename = list(uploaded.keys())[0]
        self.df = pd.read_csv(io.BytesIO(uploaded[filename]))
        self.original_df = self.df.copy()
        print(f"Successfully loaded {filename}.")

        garbage_strings = ['?', 'n/a', 'NULL', ' ', 'N/A', 'null']
        self.df.replace(garbage_strings, np.nan, inplace=True)

        for col in self.df.columns:
            try:
                numeric_col = pd.to_numeric(self.df[col], errors='coerce')
                if not numeric_col.isna().all():
                    self.df[col] = numeric_col
            except Exception:
                pass
        print("Data ingestion and auto-sanitization complete.")

    def data_summary(self):
        if self.df is None: return "No data loaded."
        print(f"--- Data Summary ---")
        print(f"Shape: {self.df.shape[0]} Rows, {self.df.shape[1]} Columns")
        print("\n--- Column Breakdown ---")
        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()
        print(f"Numerical ({len(num_cols)}): {num_cols}")
        print(f"Categorical ({len(cat_cols)}): {cat_cols}")
        print("\n--- Data Preview (First 20 Rows) ---")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='mean', fill_value=None):
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].isna().sum() > 0:
                if self.df[col].dtype in [np.float64, np.int64]:
                    if strategy == 'mean':
                        self.df[col] = self.df[col].fillna(self.df[col].mean())
                    elif strategy == 'median':
                        self.df[col] = self.df[col].fillna(self.df[col].median())
                    elif strategy == 'constant' and fill_value is not None:
                        self.df[col] = self.df[col].fillna(fill_value)
                else:
                    self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
        print(f"Missing values handled using '{strategy}' strategy.")

    def remove_duplicates(self):
        if self.df is None: return
        initial_rows = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_rows - self.df.shape[0]} duplicate rows.")

    def handle_outliers(self, column, action='flag'):
        if self.df is None or column not in self.df.columns: return

        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_condition = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)

        if action == 'drop':
            initial_rows = self.df.shape[0]
            self.df = self.df[~outlier_condition]
            print(f"Dropped {initial_rows - self.df.shape[0]} outlier rows from {column}.")
        elif action == 'flag':
            self.df[f'{column}_is_outlier'] = outlier_condition
            print(f"Flagged outliers in new column: {column}_is_outlier.")

    def delete_rows(self, row_indices_str):
        indices = [int(i.strip()) for i in row_indices_str.split(',') if i.strip().isdigit()]
        self.df.drop(index=indices, inplace=True, errors='ignore')
        print(f"Deleted rows: {indices}")

    def delete_columns(self, columns_str):
        cols = [c.strip() for c in columns_str.split(',')]
        self.df.drop(columns=cols, inplace=True, errors='ignore')
        print(f"Deleted columns: {cols}")

    def extract_normalized_numeric_data(self, strategy='standard'):
        num_cols = self.df.select_dtypes(include=np.number).columns
        df_num = self.df[num_cols].copy()

        if strategy == 'minmax':
            scaler = MinMaxScaler()
        elif strategy == 'robust':
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()

        df_num[num_cols] = scaler.fit_transform(df_num)
        return df_num

    def extract_normalized_categorical_data(self, strategy='onehot'):
        cat_cols = self.df.select_dtypes(exclude=np.number).columns
        df_cat = self.df[cat_cols].copy()

        if strategy == 'onehot':
            return pd.get_dummies(df_cat, columns=cat_cols, drop_first=True)
        elif strategy == 'ordinal' or strategy == 'uniform':
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(df_cat)
            df_encoded = pd.DataFrame(encoded, columns=cat_cols, index=df_cat.index)
            if strategy == 'uniform':
                scaler = MinMaxScaler()
                df_encoded[cat_cols] = scaler.fit_transform(df_encoded)
            return df_encoded

    def merge_encoded_data(self, num_strategy='standard', cat_strategy='onehot'):
        df_num = self.extract_normalized_numeric_data(strategy=num_strategy)
        df_cat = self.extract_normalized_categorical_data(strategy=cat_strategy)
        merged_df = pd.concat([df_num, df_cat], axis=1)
        return merged_df

    def plot_univariate_subplots(self, column):
        if column not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[column]):
            print("Please provide a valid numeric column.")
            return

        fig = make_subplots(rows=1, cols=3, subplot_titles=("Box Plot", "Index Scatter", "Histogram"))
        fig.add_trace(go.Box(x=self.df[column], name="Box", orientation='h'), row=1, col=1)
        fig.add_trace(go.Scatter(y=self.df[column], mode='markers', name="Scatter"), row=1, col=2)
        fig.add_trace(go.Histogram(x=self.df[column], name="Hist"), row=1, col=3)
        fig.update_layout(title_text=f"Univariate Analysis: {column}", height=400)
        fig.show()

    def plot_relationship(self, col1, col2):
        is_col1_num = pd.api.types.is_numeric_dtype(self.df[col1])
        is_col2_num = pd.api.types.is_numeric_dtype(self.df[col2])

        if is_col1_num and is_col2_num:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter: {col1} vs {col2}")
        elif not is_col1_num and not is_col2_num:
            counts = self.df.groupby([col1, col2]).size().reset_index(name='count')
            fig = px.bar(counts, x=col1, y="count", color=col2, barmode='group', title=f"Grouped Bar: {col1} vs {col2}")
        else:
            cat_col, num_col = (col1, col2) if not is_col1_num else (col2, col1)
            fig = px.box(self.df, x=cat_col, y=num_col, points="all", title=f"Box: {num_col} by {cat_col}")
        fig.show()

    def plot_categorical_frequency(self, column):
        if pd.api.types.is_numeric_dtype(self.df[column]): return

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'count']
        counts['percentage'] = (counts['count'] / counts['count'].sum() * 100).round(2).astype(str) + '%'

        fig = px.bar(counts, x=column, y='count', text='percentage', title=f"Frequency: {column}")
        fig.update_traces(textposition='outside')
        fig.show()

    def _cramers_v(self, x, y):
        confusion_matrix = pd.crosstab(x, y)
        chi2 = ss.chi2_contingency(confusion_matrix)[0]
        n = confusion_matrix.sum().sum()
        phi2 = chi2 / n
        r, k = confusion_matrix.shape
        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
        rcorr = r - ((r-1)**2)/(n-1)
        kcorr = k - ((k-1)**2)/(n-1)
        return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1))) if min((kcorr-1), (rcorr-1)) > 0 else 0

    def _correlation_ratio(self, categories, measurements):
        fcat, _ = pd.factorize(categories)
        cat_num = np.max(fcat)+1
        y_avg_array = np.zeros(cat_num)
        n_array = np.zeros(cat_num)
        for i in range(0,cat_num):
            cat_measures = measurements[np.argwhere(fcat == i).flatten()]
            n_array[i] = len(cat_measures)
            y_avg_array[i] = np.average(cat_measures)
        y_total_avg = np.sum(np.multiply(y_avg_array,n_array))/np.sum(n_array)
        numerator = np.sum(np.multiply(n_array,np.power(np.subtract(y_avg_array,y_total_avg),2)))
        denominator = np.sum(np.power(np.subtract(measurements,y_total_avg),2))
        if numerator == 0: return 0.0
        return np.sqrt(numerator/denominator)

    def plot_all_associations_heatmap(self):
        cols = self.df.columns
        matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

        for col1 in cols:
            for col2 in cols:
                if col1 == col2:
                    matrix.loc[col1, col2] = 1.0
                    continue

                is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
                is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])

                temp = self.df[[col1, col2]].dropna()

                if is_num1 and is_num2:
                    matrix.loc[col1, col2] = temp[col1].corr(temp[col2])
                elif not is_num1 and not is_num2:
                    matrix.loc[col1, col2] = self._cramers_v(temp[col1], temp[col2])
                else:
                    cat_col, num_col = (col1, col2) if not is_num1 else (col2, col1)
                    matrix.loc[col1, col2] = self._correlation_ratio(temp[cat_col], temp[num_col])

        fig = px.imshow(matrix.astype(float), text_auto=".2f", aspect="auto",
                        title="Unified Association Heatmap (Pearson, Cramér's V, Eta)",
                        color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
        fig.show()


if __name__ == "__main__":
    inspector = DataInspector()

    inspector.upload_data()

    if inspector.df is not None:

        inspector.data_summary()

        inspector.remove_duplicates()
        if 'Age' in inspector.df.columns:
            inspector.handle_missing_values(strategy='median')
            inspector.handle_outliers(column='Age', action='flag')

        final_machine_learning_df = inspector.merge_encoded_data()
        print("\n--- Final ML Ready DataFrame Shape ---")
        print(final_machine_learning_df.shape)

        if 'Age' in inspector.df.columns:
            inspector.plot_univariate_subplots('Age')
        if 'Age' in inspector.df.columns and 'Fare' in inspector.df.columns:
            inspector.plot_relationship('Age', 'Fare')
        if 'Survived' in inspector.df.columns and 'Fare' in inspector.df.columns:
             inspector.plot_relationship('Survived', 'Fare')

        inspector.plot_all_associations_heatmap()

Please upload your CSV file:


Saving titanic.csv to titanic (1).csv
Successfully loaded titanic (1).csv.
Data ingestion and auto-sanitization complete.
--- Data Summary ---
Shape: 891 Rows, 12 Columns

--- Column Breakdown ---
Numerical (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
Categorical (4): ['Name', 'Sex', 'Cabin', 'Embarked']

--- Data Preview (First 20 Rows) ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


Removed 0 duplicate rows.
Missing values handled using 'median' strategy.
Flagged outliers in new column: Age_is_outlier.

--- Final ML Ready DataFrame Shape ---
(891, 1048)
